## Imports and loading

In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.impute import SimpleImputer
import html
import re

print("Caricamento dati...")


train_df = pd.read_csv("development.csv", na_values=['\\N', '0000-00-00 00:00:00', '  '], encoding='utf-8')
eval_df = pd.read_csv("evaluation.csv", na_values=['\\N', '0000-00-00 00:00:00', '  '])

print(train_df.isna().sum())



Caricamento dati...
Id               0
source         294
title            3
article       1878
page_rank        0
timestamp    27750
label            0
dtype: int64


## Text cleaning

In [19]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase (sempre fondamentale per generalizzare)
    text = text.lower()
    
    # 2. Decodifica entità HTML (es. &amp; -> &)
    text = html.unescape(text)
    
    # 3. TRUCCO: Invece di rimuovere i tag con <[^>]+>, 
    # sostituiamo i caratteri sintattici dell'HTML e degli URL con spazi.
    # Così <a href="...  diventa " a href ... " e le parole restano!
    # Sostituiamo < > / . = " : _ - con spazi
    text = re.sub(r'[<>/\\\.=":_\-]', ' ', text)
    
    # 4. Rimuovi solo le parole puramente tecniche che non servono
    # es. 'http', 'www', 'href', 'com', 'jpg'
    # Puoi commentare questa parte se vuoi tenere tutto
    stop_garbage = r'\b(http|https|www|com|href|src|jpg|png|html|height|width|border)\b'
    
    # 3. GESTIONE NUMERI (La tua intuizione corretta)
    # Sostituisci i numeri con un token generico "NUM"
    # Esempio: "Oil rose by 5%" -> "Oil rose by NUM%"
    text = re.sub(r'\d+', 'NUM', text)
    
    # 6. Rimuovi singole lettere (refusi)
    text = re.sub(r'\b[a-z]\b', ' ', text)
    
    # 7. Normalizza spazi
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

train_df['title'] = train_df['title'].fillna('')
train_df['article'] = train_df['article'].fillna('')

eval_df['title'] = eval_df['title'].fillna('')
eval_df['article'] = eval_df['article'].fillna('')

train_df['title'] = train_df['title'].apply(clean_text)
eval_df['title'] = eval_df['title'].apply(clean_text)

train_df['article'] = train_df['article'].apply(clean_text)
eval_df['article'] = eval_df['article'].apply(clean_text)


train_df['text_content'] = train_df['title']+ ' ' + train_df['article']

eval_df['text_content'] = eval_df['title']+ ' ' + eval_df['article']

print("Dati caricati.")

Dati caricati.


In [20]:
print(f"Dataset originale: {len(train_df)} righe.")
train_df = train_df[train_df['text_content'] != " "]
print(f"Dataset dopo rimozione righe con title e article vuoti: {len(train_df)} righe.")


group_cols = ['text_content']

# 1. Funzione per determinare la moda o segnare il pareggio
# Se len(x.mode()) == 1, c'è una maggioranza chiara -> restituisce la moda.
# Altrimenti (len > 1), c'è un pareggio -> restituisce -1.
def get_majority_or_tie(x):
    modes = x.mode()
    if len(modes) == 1:
        return modes[0]
    else:
        return -1


# 2. Applica la logica
voted_labels = train_df.groupby(group_cols)['label'].transform(get_majority_or_tie)

# 3. Sovrascrivi la colonna label
train_df['label'] = voted_labels

# 4. Elimina PRIMA i pareggi (tutte le righe con -1 vengono scartate)
# Assicurati che -1 non sia una label valida nel tuo dataset originale!
train_df = train_df[train_df['label'] != -1]

# 5. Elimina i duplicati
# Ora rimangono solo gli articoli con una maggioranza chiara.
# Possiamo tenere la prima riga di ogni gruppo.
train_df = train_df.drop_duplicates(subset=group_cols, keep='first')

print(f"Dataset pulito: {len(train_df)} righe (pareggi eliminati).")




group_cols = ['article']

# 1. CREA UNA MASCHERA per separare i dati
# True per le righe da elaborare, False per quelle da ignorare (title vuoto)
mask_process = train_df['article'].str.strip() != ''

# 2. SEPARA IL DATAFRAME IN DUE PEZZI
df_to_process = train_df[mask_process].copy() # Righe con titolo valido
df_ignored = train_df[~mask_process].copy()   # Righe con title='' (da conservare intatte)

# 3. APPLICA LA LOGICA SOLO SU 'df_to_process'
if not df_to_process.empty:
    # Calcola la maggioranza
    voted_labels = df_to_process.groupby(group_cols)['label'].transform(get_majority_or_tie)
    
    # Assegna le nuove label
    df_to_process['label'] = voted_labels
    
    # Elimina i pareggi (-1)
    df_to_process = df_to_process[df_to_process['label'] != -1]
    
    # Elimina i duplicati (mantenendo una sola riga per titolo)
    df_to_process = df_to_process.drop_duplicates(subset=group_cols, keep='first')

# 4. RIUNISCI I DUE PEZZI
# Concateniamo le righe elaborate (pulite e deduplicate) con quelle ignorate (originali)
train_df = pd.concat([df_to_process, df_ignored], ignore_index=True)

print(f"Dataset finale dopo majority voting su 'article': {len(train_df)} righe.")



Dataset originale: 79997 righe.
Dataset dopo rimozione righe con title e article vuoti: 79996 righe.
Dataset pulito: 75496 righe (pareggi eliminati).
Dataset finale dopo majority voting su 'article': 74529 righe.


## Preprocessing

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder


text_transformer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    )


cat_transformer = OneHotEncoder(handle_unknown='ignore')

hour_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ])
source_pipe = Pipeline([
    ('imputer',SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(max_categories=51, handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        ('ord', cat_transformer, ['page_rank']),
        ('hour', hour_transformer, ['hour_cos','hour_sin']),
        ('cat_source', source_pipe, ['source'])        
    ]
)
train_df['day_of_week'] = pd.to_datetime(train_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.dayofweek
eval_df['day_of_week'] = pd.to_datetime(eval_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.dayofweek

train_df['is_weekend'] = train_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
train_df['timestamp_missing'] = train_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)
eval_df['is_weekend'] = eval_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
eval_df['timestamp_missing'] = eval_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)

train_df['text_length'] = train_df['text_content'].apply(lambda x: len(x))
eval_df['text_length'] = eval_df['text_content'].apply(lambda x: len(x))

train_df['hour_of_day'] = pd.to_datetime(train_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.hour
eval_df['hour_of_day'] = pd.to_datetime(eval_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.hour
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour_of_day'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour_of_day'] / 24)

eval_df['hour_sin'] = np.sin(2 * np.pi * eval_df['hour_of_day'] / 24)
eval_df['hour_cos'] = np.cos(2 * np.pi * eval_df['hour_of_day'] / 24)

X_eval = eval_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]


X = train_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]
y = train_df['label']


## Linear SVC

In [54]:

 ## LINEAR SVC MODEL

classifier = LinearSVC(random_state=42)
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])


param_grid = {
    'clf__C': [0.1],
    'clf__class_weight': ['balanced'],
    'preprocessor__txt__ngram_range': [(1,3)],
    'preprocessor__txt__max_df': [0.5],
    'preprocessor__txt__max_features': [1000000],
    'preprocessor__cat_source__encoder__max_categories': [300],
    'preprocessor__txt__stop_words': ['english'],
    'preprocessor__txt__sublinear_tf': [True],
    'preprocessor__txt__lowercase': [True]
}

grid_search = GridSearchCV(
    model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_



Fitting 5 folds for each of 1 candidates, totalling 5 fits
LINEAR SVC - Grid Search Results:
{'clf__C': 0.1, 'clf__class_weight': 'balanced', 'preprocessor__cat_source__encoder__max_categories': 300, 'preprocessor__txt__lowercase': True, 'preprocessor__txt__max_df': 0.5, 'preprocessor__txt__max_features': 1000000, 'preprocessor__txt__ngram_range': (1, 3), 'preprocessor__txt__stop_words': 'english', 'preprocessor__txt__sublinear_tf': True}
Best Cross-Validation Macro F1: 0.7212


In [52]:
best_model.fit(X, y)
y_eval_pred = best_model.predict(X_eval)

pd.DataFrame({
    'Id': eval_df['Id'],
    'Predicted': y_eval_pred}).to_csv('submission.csv', index=False)

# Logistic Regression

In [56]:
from sklearn.linear_model import LogisticRegression
classifier=LogisticRegression(random_state=42)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])

param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__class_weight': ['balanced'],
    'preprocessor__txt__ngram_range': [(1,3)],
    'preprocessor__txt__max_df': [0.2, 0.3, 0.4, 0.5],
    'preprocessor__txt__min_df':[2, 3, 4, 5, 6],
    'preprocessor__cat_source__encoder__max_categories': [200, 300, 400, 500, 700],
    'preprocessor__txt__stop_words': ['english'],
    'preprocessor__txt__sublinear_tf': [True],
    'preprocessor__txt__lowercase': [True]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2)

grid_search.fit(X, y)

print("LOGISTIC REGRESSION - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


KeyboardInterrupt: 

# GRAFICO

In [ ]:
from matplotlib import pyplot as plt


max_features = np.arange(50000, 3500001, 100000)

svm= LinearSVC(class_weight='balanced', random_state=42)
logreg= LogisticRegression(class_weight='balanced', random_state=42)

model_pipeline_svm = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', svm)
])
model_pipeline_logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', logreg)
])

scores_lr = []
scores_svm = []

for max_feat in max_features:
    print(f"Testing max_features={max_feat}...")
    param_grid = {
        'preprocessor__txt__ngram_range': [(1,3)],
        'preprocessor__txt__max_features': [max_feat],
        'preprocessor__cat_source__encoder__max_categories': [300],
        'preprocessor__txt__stop_words': ['english'],
        'preprocessor__txt__sublinear_tf': [True],
        'preprocessor__txt__lowercase': [True]
    }
    grid_search_svm = GridSearchCV(
        model_pipeline_svm,
        param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    grid_search_logreg = GridSearchCV(
        model_pipeline_logreg,
        param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    
    grid_search_svm.fit(X, y)
    scores_svm.append(grid_search_svm.best_score_)
    grid_search_logreg.fit(X, y)
    scores_lr.append(grid_search_logreg.best_score_)
    
plt.figure(figsize=(8, 6))

# Plot delle linee
plt.plot(max_features, scores_lr, label='Logistic Regression', marker='')
plt.plot(max_features, scores_svm, label='SVM', marker='')

# Etichette e Stile
plt.xlabel('n (max_features)', fontsize=14)
plt.ylabel('macro $f_1$ score', fontsize=14)
plt.grid(True, alpha=0.6)
plt.legend(fontsize=12)

# Scala logaritmica sull'asse X spesso aiuta a vedere meglio se spazi molto i valori
plt.xscale('log') 

# Limiti asse Y per focalizzare (opzionale, adatta ai tuoi risultati)
plt.ylim(0.6, 1.0) 

plt.show()
    

Testing max_features=50000...
